# Stock Pattern Detection with SQL MATCH_RECOGNIZE

This notebook demonstrates how to use SQL **MATCH_RECOGNIZE** for detecting trading patterns in stock price data. MATCH_RECOGNIZE is a powerful SQL feature that lets you write regex-like patterns to find sequences of events across ordered rows.

**What you'll learn:**
* Generate realistic stock price time series data
* Detect bullish and bearish trends
* Identify reversal patterns (V-shaped recoveries)
* Find technical analysis patterns (double bottoms)
* Spot high volatility periods
* Calculate meaningful trading metrics (% gains, drawdowns, pattern duration)

**Use cases:** Trading signal detection, anomaly detection in sensor data, user behavior analysis, fraud detection, and any sequential pattern matching task.

In [0]:
%sql
-- Generate 60 days of stock prices with intentional patterns
CREATE OR REPLACE TEMPORARY VIEW stock_prices AS
WITH RECURSIVE date_range AS (
  SELECT 1 as day_num, DATE '2024-01-01' as trade_date
  UNION ALL
  SELECT day_num + 1, DATE_ADD(trade_date, 1)
  FROM date_range
  WHERE day_num < 60
),
aapl_prices AS (
  SELECT 
    'AAPL' as ticker,
    day_num,
    trade_date,
    CASE 
      -- Days 1-6: Bullish run (rising trend)
      WHEN day_num BETWEEN 1 AND 6 THEN 150.0 + (day_num * 3.5)
      -- Days 7-8: Small pullback
      WHEN day_num = 7 THEN 169.0
      WHEN day_num = 8 THEN 167.5
      -- Days 9-14: Strong bullish run
      WHEN day_num BETWEEN 9 AND 14 THEN 167.5 + ((day_num - 8) * 4.2)
      -- Days 15-20: Bearish correction (falling)
      WHEN day_num BETWEEN 15 AND 20 THEN 192.7 - ((day_num - 14) * 3.8)
      -- Days 21-27: V-shaped recovery (down then up)
      WHEN day_num BETWEEN 21 AND 23 THEN 169.9 - ((day_num - 20) * 4.5)
      WHEN day_num BETWEEN 24 AND 27 THEN 156.4 + ((day_num - 23) * 5.2)
      -- Days 28-35: Double bottom pattern
      WHEN day_num = 28 THEN 177.2
      WHEN day_num = 29 THEN 174.5
      WHEN day_num = 30 THEN 171.0  -- First bottom
      WHEN day_num = 31 THEN 174.0
      WHEN day_num = 32 THEN 177.5
      WHEN day_num = 33 THEN 175.0
      WHEN day_num = 34 THEN 172.0  -- Second bottom (similar level)
      WHEN day_num = 35 THEN 176.0
      -- Days 36-42: High volatility (whipsaw)
      WHEN day_num = 36 THEN 179.0
      WHEN day_num = 37 THEN 174.5
      WHEN day_num = 38 THEN 180.0
      WHEN day_num = 39 THEN 175.0
      WHEN day_num = 40 THEN 181.5
      WHEN day_num = 41 THEN 176.5
      WHEN day_num = 42 THEN 182.0
      -- Days 43-60: Gradual uptrend with noise
      ELSE 182.0 + ((day_num - 42) * 1.8) + ((day_num % 5) - 2.0)
    END as close_price
  FROM date_range
),
tsla_prices AS (
  SELECT 
    'TSLA' as ticker,
    day_num,
    trade_date,
    CASE 
      -- Days 1-5: Starting consolidation
      WHEN day_num BETWEEN 1 AND 5 THEN 200.0 + ((day_num % 3) - 1.0)
      -- Days 6-11: Bullish breakout
      WHEN day_num BETWEEN 6 AND 11 THEN 200.0 + ((day_num - 5) * 5.5)
      -- Days 12-17: Sharp bearish drop
      WHEN day_num BETWEEN 12 AND 17 THEN 233.0 - ((day_num - 11) * 4.5)
      -- Days 18-24: V-recovery
      WHEN day_num BETWEEN 18 AND 20 THEN 206.0 - ((day_num - 17) * 3.0)
      WHEN day_num BETWEEN 21 AND 24 THEN 197.0 + ((day_num - 20) * 6.0)
      -- Days 25-45: Choppy sideways (high volatility)
      WHEN day_num BETWEEN 25 AND 45 THEN 215.0 + (((day_num * 7) % 17) - 8.0)
      -- Days 46-60: Final rally
      WHEN day_num BETWEEN 46 AND 60 THEN 215.0 + ((day_num - 45) * 3.2)
      ELSE 250.0
    END as close_price
  FROM date_range
)
SELECT * FROM aapl_prices
UNION ALL
SELECT * FROM tsla_prices
ORDER BY ticker, day_num;

-- Preview the data
SELECT * FROM stock_prices LIMIT 10;

## 1. Bullish Run Detection

**What traders look for:** A sustained upward price movement over multiple consecutive days indicates strong buying pressure and positive momentum. Traders often enter positions during bullish runs or wait for pullbacks to buy.

**Why it matters:** 
* Confirms trend strength and investor confidence
* Can signal breakout from consolidation
* Helps identify entry points for trend-following strategies
* 4+ consecutive up days is statistically significant

**Pattern:** `RISING{4,}` — matches 4 or more consecutive days where each close is higher than the previous day.

In [0]:
%sql
SELECT 
  ticker,
  start_date,
  end_date,
  start_price,
  end_price,
  days_rising,
  price_gain,
  ROUND((price_gain / start_price) * 100, 2) as gain_pct,
  ROUND(price_gain / days_rising, 2) as avg_daily_gain
FROM stock_prices
  MATCH_RECOGNIZE (
    PARTITION BY ticker
    ORDER BY day_num
    MEASURES
      FIRST(trade_date) AS start_date,
      LAST(trade_date) AS end_date,
      FIRST(close_price) AS start_price,
      LAST(close_price) AS end_price,
      COUNT(*) AS days_rising,
      LAST(close_price) - FIRST(close_price) AS price_gain
    PATTERN (RISING{4,})
    DEFINE
      RISING AS close_price > LAG(close_price)
  )
ORDER BY ticker, start_date;

## 2. Bearish Run Detection

**What traders look for:** Sustained downward movement signals selling pressure, profit-taking, or negative sentiment. Multiple consecutive down days can indicate:
* Trend reversal from bullish to bearish
* Correction within a larger uptrend
* Capitulation phase (panic selling)

**Why it matters:**
* Risk management — helps identify when to exit or tighten stops
* Contrarian opportunity — oversold conditions may present buying opportunities
* Validates bearish thesis for short sellers

**Pattern:** `FALLING{4,}` — matches 4 or more consecutive declining closes.

In [0]:
%sql
SELECT 
  ticker,
  start_date,
  end_date,
  start_price,
  end_price,
  days_falling,
  price_loss,
  ROUND((price_loss / start_price) * 100, 2) as loss_pct,
  ROUND(price_loss / days_falling, 2) as avg_daily_loss
FROM stock_prices
  MATCH_RECOGNIZE (
    PARTITION BY ticker
    ORDER BY day_num
    MEASURES
      FIRST(trade_date) AS start_date,
      LAST(trade_date) AS end_date,
      FIRST(close_price) AS start_price,
      LAST(close_price) AS end_price,
      COUNT(*) AS days_falling,
      FIRST(close_price) - LAST(close_price) AS price_loss
    PATTERN (FALLING{4,})
    DEFINE
      FALLING AS close_price < LAG(close_price)
  )
ORDER BY ticker, start_date;

## 3. V-Shaped Recovery Pattern

**What traders look for:** A sharp decline followed immediately by a strong rally forms a "V" shape on the chart. This pattern suggests:
* Panic selling followed by bargain hunting
* False breakdown or temporary fear
* Strong underlying support level
* Rapid sentiment reversal

**Why it matters:**
* **High conviction signal** — violent reversals often indicate strong institutional buying
* **Risk/reward opportunity** — catching the recovery can yield quick profits
* **Trend confirmation** — suggests the prior uptrend remains intact
* **Stop-loss trap** — many traders get stopped out at the bottom, missing the recovery

**Pattern:** `DOWN{3,} UP{3,}` — at least 3 consecutive down days immediately followed by at least 3 consecutive up days.

In [0]:
%sql
-- V-shaped recovery: find bearish runs followed by bullish runs
WITH bearish_periods AS (
  SELECT 
    ticker,
    start_date,
    end_date as trough_date,
    start_price as peak_price,
    end_price as trough_price,
    days as down_days,
    end_day as trough_day
  FROM stock_prices
    MATCH_RECOGNIZE (
      PARTITION BY ticker
      ORDER BY day_num
      MEASURES
        FIRST(trade_date) AS start_date,
        LAST(trade_date) AS end_date,
        FIRST(close_price) AS start_price,
        LAST(close_price) AS end_price,
        COUNT(*) AS days,
        LAST(day_num) AS end_day
      PATTERN (FALLING{3,})
      DEFINE
        FALLING AS close_price < LAG(close_price)
    )
),
bullish_periods AS (
  SELECT 
    ticker,
    start_date as recovery_start,
    end_date as recovery_date,
    start_price as recovery_start_price,
    end_price as recovery_price,
    days as up_days,
    start_day as recovery_start_day
  FROM stock_prices
    MATCH_RECOGNIZE (
      PARTITION BY ticker
      ORDER BY day_num
      MEASURES
        FIRST(trade_date) AS start_date,
        LAST(trade_date) AS end_date,
        FIRST(close_price) AS start_price,
        LAST(close_price) AS end_price,
        COUNT(*) AS days,
        FIRST(day_num) AS start_day
      PATTERN (RISING{3,})
      DEFINE
        RISING AS close_price > LAG(close_price)
    )
)
SELECT 
  b.ticker,
  b.start_date as peak_date,
  b.trough_date,
  bull.recovery_date,
  b.peak_price,
  b.trough_price,
  bull.recovery_price,
  b.down_days,
  bull.up_days,
  (b.down_days + bull.up_days) as total_days,
  (b.peak_price - b.trough_price) as drawdown,
  ROUND(((b.peak_price - b.trough_price) / b.peak_price) * 100, 2) as drawdown_pct,
  (bull.recovery_price - b.trough_price) as recovery_gain,
  ROUND(((bull.recovery_price - b.trough_price) / b.trough_price) * 100, 2) as recovery_pct,
  CASE 
    WHEN bull.recovery_price >= b.peak_price THEN 'Full Recovery+'
    WHEN ((bull.recovery_price - b.trough_price) / b.trough_price) * 100 >= 50 THEN 'Strong Recovery'
    ELSE 'Partial Recovery'
  END as recovery_strength
FROM bearish_periods b
JOIN bullish_periods bull
  ON b.ticker = bull.ticker
  AND bull.recovery_start_day = b.trough_day + 1  -- Bullish run starts immediately after bearish
ORDER BY b.ticker, b.start_date;

## 4. Double Bottom Pattern (Technical Analysis)

**What traders look for:** A classic reversal pattern where price tests a support level twice, forming two "bottoms" at similar price levels, separated by a modest rally. The pattern completes when price breaks above the middle peak.

**Structure:**
1. First decline to a low (bottom 1)
2. Rally back up
3. Second decline to a similar low (bottom 2)
4. Final rally breaking the middle peak

**Why it matters:**
* **High-probability reversal** — one of the most reliable bullish patterns
* **Clear risk/reward** — stop below the double bottom, target above the middle peak
* **Volume confirmation** — second bottom should show less volume (weaker selling)
* **Support validation** — twice-tested support is stronger

**Pattern:** `DOWN{2,} UP{2,} DOWN{2,} UP{2,}` with similar bottom prices (within 2% tolerance).

In [0]:
%sql
-- Note: Double bottom requires comparing two separate bottoms, which is complex with MATCH_RECOGNIZE limitations
-- This simplified version finds patterns where price dips twice to similar levels
-- Using window functions to find local minima (bottoms)
WITH price_with_neighbors AS (
  SELECT 
    ticker,
    day_num,
    trade_date,
    close_price,
    LAG(close_price, 1) OVER (PARTITION BY ticker ORDER BY day_num) as prev_price,
    LAG(close_price, 2) OVER (PARTITION BY ticker ORDER BY day_num) as prev2_price,
    LEAD(close_price, 1) OVER (PARTITION BY ticker ORDER BY day_num) as next_price,
    LEAD(close_price, 2) OVER (PARTITION BY ticker ORDER BY day_num) as next2_price
  FROM stock_prices
),
local_bottoms AS (
  SELECT 
    ticker,
    day_num,
    trade_date,
    close_price,
    ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY day_num) as bottom_num
  FROM price_with_neighbors
  WHERE close_price < prev_price 
    AND close_price < prev2_price
    AND close_price < next_price
    AND close_price < next2_price
),
double_bottoms AS (
  SELECT 
    b1.ticker,
    b1.trade_date as first_bottom_date,
    b2.trade_date as second_bottom_date,
    b1.close_price as first_bottom_price,
    b2.close_price as second_bottom_price,
    ABS(b2.close_price - b1.close_price) as bottom_difference,
    b2.day_num - b1.day_num as days_between
  FROM local_bottoms b1
  JOIN local_bottoms b2
    ON b1.ticker = b2.ticker
    AND b2.bottom_num = b1.bottom_num + 1
    AND b2.day_num - b1.day_num BETWEEN 4 AND 15  -- Bottoms 4-15 days apart
    AND ABS(b2.close_price - b1.close_price) / b1.close_price <= 0.03  -- Within 3%
)
SELECT 
  ticker,
  first_bottom_date,
  second_bottom_date,
  first_bottom_price,
  second_bottom_price,
  bottom_difference,
  ROUND((bottom_difference / first_bottom_price) * 100, 2) as bottom_diff_pct,
  days_between,
  CASE 
    WHEN (bottom_difference / first_bottom_price) * 100 <= 1 THEN 'Textbook Double Bottom'
    WHEN (bottom_difference / first_bottom_price) * 100 <= 2 THEN 'Valid Double Bottom'
    ELSE 'Loose Double Bottom'
  END as pattern_quality
FROM double_bottoms
ORDER BY ticker, first_bottom_date;

## 5. High Volatility Period Detection

**What traders look for:** Periods of rapid price oscillation where direction alternates frequently ("whipsaw" or "choppy" market). Characteristics:
* No clear trend — frequent reversals
* Large intraday or day-to-day swings
* Conflicting signals and false breakouts

**Why it matters:**
* **Risk management** — trend-following strategies get chopped up; reduce position sizes
* **Strategy adjustment** — favor range-bound strategies (sell resistance, buy support)
* **Option opportunity** — high volatility increases option premiums (good for sellers)
* **Wait for clarity** — often better to stay flat until trend emerges

**Pattern:** `(UP DOWN){3,}` — at least 3 consecutive up-down alternating pairs (6+ days of back-and-forth).

In [0]:
%sql
-- High volatility: use rolling window to find periods with many direction changes
WITH daily_changes AS (
  SELECT 
    ticker,
    day_num,
    trade_date,
    close_price,
    close_price - LAG(close_price) OVER (PARTITION BY ticker ORDER BY day_num) as price_change,
    CASE 
      WHEN close_price > LAG(close_price) OVER (PARTITION BY ticker ORDER BY day_num) THEN 1
      WHEN close_price < LAG(close_price) OVER (PARTITION BY ticker ORDER BY day_num) THEN -1
      ELSE 0
    END as direction
  FROM stock_prices
),
direction_changes AS (
  SELECT 
    ticker,
    day_num,
    trade_date,
    close_price,
    price_change,
    direction,
    LAG(direction) OVER (PARTITION BY ticker ORDER BY day_num) as prev_direction,
    CASE 
      WHEN direction != LAG(direction) OVER (PARTITION BY ticker ORDER BY day_num) 
           AND direction != 0 AND LAG(direction) OVER (PARTITION BY ticker ORDER BY day_num) != 0
      THEN 1 
      ELSE 0 
    END as is_change
  FROM daily_changes
),
rolling_volatility AS (
  SELECT 
    ticker,
    trade_date,
    close_price,
    SUM(is_change) OVER (PARTITION BY ticker ORDER BY day_num ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as changes_in_7_days,
    MIN(close_price) OVER (PARTITION BY ticker ORDER BY day_num ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as min_price_7d,
    MAX(close_price) OVER (PARTITION BY ticker ORDER BY day_num ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as max_price_7d,
    FIRST_VALUE(trade_date) OVER (PARTITION BY ticker ORDER BY day_num ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as window_start_date,
    ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY day_num) as row_num
  FROM direction_changes
)
SELECT 
  ticker,
  window_start_date as start_date,
  trade_date as end_date,
  min_price_7d as low_price,
  max_price_7d as high_price,
  changes_in_7_days as direction_changes,
  (max_price_7d - min_price_7d) as price_range,
  ROUND(((max_price_7d - min_price_7d) / min_price_7d) * 100, 2) as range_pct,
  CASE 
    WHEN ((max_price_7d - min_price_7d) / min_price_7d) * 100 >= 15 THEN 'Extreme Volatility'
    WHEN ((max_price_7d - min_price_7d) / min_price_7d) * 100 >= 10 THEN 'High Volatility'
    ELSE 'Moderate Volatility'
  END as volatility_level
FROM rolling_volatility
WHERE row_num >= 7  -- Only show complete 7-day windows
  AND changes_in_7_days >= 4  -- At least 4 direction changes in 7 days
ORDER BY ticker, start_date;

## Key Takeaways

### MATCH_RECOGNIZE Syntax Components

1. **PARTITION BY** — Groups data by category (like window functions)
2. **ORDER BY** — Defines the sequence for pattern matching
3. **MEASURES** — Calculates output columns from matched rows
4. **PATTERN** — The regex-like pattern definition
   * `{n,}` = n or more occurrences
   * `{n,m}` = between n and m occurrences
   * `|` = alternation (or)
   * `()` = grouping
5. **DEFINE** — Boolean conditions for each pattern variable
   * Use `LAG()`, `FIRST()`, `LAST()` to reference other rows
   * Access pattern variable columns with dot notation (e.g., `DOWN.close_price`)

### Trading Pattern Insights

* **Bullish/Bearish Runs** — Trend confirmation; longer runs = stronger momentum
* **V-Shaped Recoveries** — High-conviction reversal signals; quick profit opportunities
* **Double Bottoms** — Reliable reversal pattern; clear risk/reward setup
* **High Volatility** — Adjust strategy; reduce size or switch to range-trading

### Practical Applications Beyond Trading

* **IoT/Sensor Data** — Detect equipment degradation patterns, anomaly sequences
* **User Behavior** — Find engagement patterns, churn signals, conversion paths
* **Fraud Detection** — Identify suspicious transaction sequences
* **Supply Chain** — Spot inventory depletion patterns, demand spikes
* **Healthcare** — Detect vital sign deterioration patterns, recovery trajectories

### Next Steps

1. **Apply to real tables** — Replace the temp view with your actual data
2. **Tune pattern parameters** — Adjust thresholds (e.g., `{3,}` vs `{5,}`)
3. **Combine patterns** — Create complex multi-stage detection logic
4. **Add actions** — Trigger alerts, create views, populate downstream tables
5. **Optimize performance** — Use appropriate partitioning and indexes